In [0]:
# -------------------------------------------------------------------------
# 1. BRONZE LAYER: Ingesting Raw Data Streams into Cloud Memory
# -------------------------------------------------------------------------
from pyspark.sql import functions as F

print("Initializing Bronze Layer Data Ingestion...")

Initializing Bronze Layer Data Ingestion...


In [0]:
students_data = [(101, "Guru Prasad"), (102, "Ananya Rao"), (104, "Siddharth Singh"), 
                 (105, "Meghana Reddy"), (106, "Kiran Rao"), (107, "Vikram Malhotra")]
df_students = spark.createDataFrame(students_data, ["student_id", "student_name"])

In [0]:
courses_data = [("CS-101", "Deep Learning"), ("CS-102", "Azure Cloud"), ("CS-103", "MERN Stack")]
df_courses = spark.createDataFrame(courses_data, ["course_id", "course_name"])

In [0]:
enrollments_data = [(1001, 101, "CS-101", "Active"), (1002, 102, "CS-102", "Completed"),
                    (1003, 104, "CS-103", "Dropped"), (1004, 105, "CS-102", "Active"),
                    (1005, 106, "CS-101", "Completed"), (1006, 107, "CS-103", "Active")]
df_enrollments = spark.createDataFrame(enrollments_data, ["enrollment_id", "student_id", "course_id", "status"])

In [0]:
progress_data = [(1001, 45.0), (1002, 115.0), (1003, 0.0), (1004, -15.0), (1005, 100.0), (1006, 64.0)]
df_progress = spark.createDataFrame(progress_data, ["enrollment_id", "completion_percentage"])

In [0]:
# -------------------------------------------------------------------------
# 2. SILVER LAYER: Data Cleansing & Master Table Join
# -------------------------------------------------------------------------
print("Executing Silver Layer Data Cleansing and Distributed Joins...")

# Apply the Python boundary-clipping business rules dynamically at scale using Spark
df_progress_clean = df_progress.withColumn(
    "completion_percentage",
    F.when(F.col("completion_percentage") > 100.0, 100.0)
     .when(F.col("completion_percentage") < 0.0, 0.0)
     .otherwise(F.col("completion_percentage"))
)

Executing Silver Layer Data Cleansing and Distributed Joins...


In [0]:
df_silver_master = df_enrollments \
    .join(df_students, "student_id", "inner") \
    .join(df_courses, "course_id", "inner") \
    .join(df_progress_clean, "enrollment_id", "inner") \
    .withColumn("ingested_at_time", F.current_timestamp())

# Save the unified cleaned data as a permanent, high-performance Delta Table
df_silver_master.write.format("delta").mode("overwrite").saveAsTable("silver_course_progress_master")

In [0]:
df_silver_master.write.format("delta").mode("overwrite").saveAsTable("silver_course_progress_master")

In [0]:
# -------------------------------------------------------------------------
# 3. GOLD LAYER: Aggregating KPIs for Business Dashboards
# -------------------------------------------------------------------------
print("Compiling Gold Layer Analytical Metrics Report...")

df_gold_metrics = spark.table("silver_course_progress_master") \
    .groupby("course_id", "course_name").agg(
        F.count("student_id").alias("Total_Enrolled"),
        F.round(F.avg("completion_percentage"), 2).alias("Average_Progress"),
        F.sum(F.when(F.col("status") == "Completed", 1).otherwise(0)).alias("Total_Completions"),
        F.sum(F.when(F.col("status") == "Dropped", 1).otherwise(0)).alias("Total_Dropouts")
    ).orderBy("course_id")

Compiling Gold Layer Analytical Metrics Report...


In [0]:
df_gold_metrics.write.format("delta").mode("overwrite").saveAsTable("gold_course_performance_summary")

print("Medallion Architecture tables successfully populated in Delta Lake!")

Medallion Architecture tables successfully populated in Delta Lake!


In [0]:
%sql
-- Compact the small storage files to maximize cloud query speeds
OPTIMIZE gold_course_performance_summary ZORDER BY (course_id);

-- Query your table's historical delta transaction ledger log
DESCRIBE HISTORY gold_course_performance_summary;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-21T08:42:40.000Z,147985412235563,azuser7213_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(283909929956797),561b25b4-54d0-4e48-a0ad-85d750ffe447,0621-083810-wzt9fomd-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 3, numOutputBytes -> 2000)",null,Databricks-Runtime/18.2.x-photon-scala2.13
